# Export ke ONNX FP32

In [1]:
# step1_export_fp32.py
# Jalankan: python step1_export_fp32.py

import torch
import os
import sys
import importlib.util

APP_DIR   = r"C:\kuliah-sementara\SKRIPSI\streamlit_app"
SWIN_PATH = os.path.join(APP_DIR, "models", "SWIN_BEST.pth")
ONNX_FP32 = os.path.join(APP_DIR, "models", "swin_best_fp32.onnx")

# ── Auto-detect inference module ──
def load_inference_module(app_dir):
    for fname in ["inference.py", "inference-2.py"]:
        fpath = os.path.join(app_dir, fname)
        if os.path.exists(fpath):
            print(f"[INFO] Menggunakan: {fname}")
            spec = importlib.util.spec_from_file_location("inference_mod", fpath)
            mod  = importlib.util.module_from_spec(spec)
            sys.path.insert(0, app_dir)
            spec.loader.exec_module(mod)
            return mod
    raise FileNotFoundError("Tidak ditemukan inference.py atau inference-2.py")

inf_mod              = load_inference_module(APP_DIR)
SwinFeatureExtractor = inf_mod.SwinFeatureExtractor

# ── Load model ──
device = torch.device('cpu')  # WAJIB cpu untuk ONNX export
ckpt   = torch.load(SWIN_PATH, map_location=device, weights_only=False)
state  = ckpt.get('model_state_dict', ckpt)

head_out_dim = None
for key in ['head.weight', 'head.fc.weight', 'model.head.weight']:
    if key in state:
        head_out_dim = int(state[key].shape[0])
        print(f"[INFO] Head terdeteksi: '{key}' → {head_out_dim}-dim")
        break

if head_out_dim is None:
    print("[WARN] Head tidak terdeteksi, fallback ke 512")
    head_out_dim = 512

model = SwinFeatureExtractor(head_out_dim=head_out_dim).to(device)
missing, unexpected = model.load_state_dict(state, strict=False)
if missing:    print(f"[WARN] Missing keys   : {missing[:3]}")
if unexpected: print(f"[WARN] Unexpected keys: {unexpected[:3]}")
model.eval()
print("[INFO] Model loaded ✓")

# ── Export ke ONNX FP32 ──
dummy = torch.zeros(2, 3, 224, 224)  # batch=2 dual-eye
print("[INFO] Exporting ke ONNX FP32...")
with torch.no_grad():
    torch.onnx.export(
        model, dummy, ONNX_FP32,
        opset_version=17,
        input_names=["eye_crops"],
        output_names=["features"],
        dynamic_axes={
            "eye_crops": {0: "batch"},
            "features":  {0: "batch"}
        },
        verbose=False
    )

fp32_mb = os.path.getsize(ONNX_FP32) / 1e6
print(f"[INFO] FP32 tersimpan: {fp32_mb:.1f} MB")
print(f"[INFO] Path: {ONNX_FP32}")
print("[DONE] Step 1 selesai. Lanjut jalankan step2_quantize_int8.py")

[INFO] Menggunakan: inference.py


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

[INFO] GPU: NVIDIA GeForce RTX 3060  |  VRAM: 12.9 GB
[INFO] AMP float16 aktif — inferensi lebih cepat & hemat VRAM
[WARN] Head tidak terdeteksi, fallback ke 512
[INFO] Swin: skema A (timm head, 512-dim feature)
[INFO] Swin backbone output: 512-dim, final feature: 512-dim
[WARN] Missing keys   : ['backbone.head.fc.weight', 'backbone.head.fc.bias']
[WARN] Unexpected keys: ['classifier.1.weight', 'classifier.1.bias', 'classifier.4.weight']
[INFO] Model loaded ✓
[INFO] Exporting ke ONNX FP32...


c:\Users\USER\miniconda3\envs\skripsi_env\lib\site-packages\torch\__init__.py:2150: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert condition, message


[INFO] FP32 tersimpan: 114.5 MB
[INFO] Path: C:\kuliah-sementara\SKRIPSI\streamlit_app\models\swin_best_fp32.onnx
[DONE] Step 1 selesai. Lanjut jalankan step2_quantize_int8.py


# Quantize FP32 → INT8

In [2]:
# step2_quantize_int8.py
# Jalankan: python step2_quantize_int8.py
# Prasyarat: step1_export_fp32.py sudah berhasil dijalankan

import os
from onnxruntime.quantization import quantize_dynamic, QuantType

APP_DIR   = r"C:\kuliah-sementara\SKRIPSI\streamlit_app"
ONNX_FP32 = os.path.join(APP_DIR, "models", "swin_best_fp32.onnx")
ONNX_INT8 = os.path.join(APP_DIR, "models", "swin_best_int8.onnx")

# Cek FP32 sudah ada
if not os.path.exists(ONNX_FP32):
    raise FileNotFoundError(
        f"File FP32 tidak ditemukan: {ONNX_FP32}\n"
        f"Jalankan step1_export_fp32.py terlebih dahulu!"
    )

fp32_mb = os.path.getsize(ONNX_FP32) / 1e6
print(f"[INFO] Input FP32: {fp32_mb:.1f} MB")
print("[INFO] Memulai INT8 quantization (dynamic)...")

quantize_dynamic(
    model_input=ONNX_FP32,
    model_output=ONNX_INT8,
    weight_type=QuantType.QInt8,
    op_types_to_quantize=['MatMul', 'Gemm']
)

int8_mb = os.path.getsize(ONNX_INT8) / 1e6
print(f"[INFO] INT8 tersimpan: {int8_mb:.1f} MB")
print(f"[INFO] Path: {ONNX_INT8}")
print(f"[INFO] Kompresi: {fp32_mb:.1f} MB → {int8_mb:.1f} MB "
      f"({(1 - int8_mb/fp32_mb)*100:.0f}% lebih kecil)")
print("[DONE] Step 2 selesai. Lanjut jalankan step3_verify_benchmark.py")

[INFO] Input FP32: 114.5 MB
[INFO] Memulai INT8 quantization (dynamic)...
[INFO] INT8 tersimpan: 31.4 MB
[INFO] Path: C:\kuliah-sementara\SKRIPSI\streamlit_app\models\swin_best_int8.onnx
[INFO] Kompresi: 114.5 MB → 31.4 MB (73% lebih kecil)
[DONE] Step 2 selesai. Lanjut jalankan step3_verify_benchmark.py


#  Verifikasi + Benchmark


In [3]:
# step3_verify_benchmark.py
# Jalankan: python step3_verify_benchmark.py
# Prasyarat: step2_quantize_int8.py sudah berhasil dijalankan

import os
import time
import numpy as np
import onnxruntime as ort

APP_DIR   = r"C:\kuliah-sementara\SKRIPSI\streamlit_app"
ONNX_INT8 = os.path.join(APP_DIR, "models", "swin_best_int8.onnx")

if not os.path.exists(ONNX_INT8):
    raise FileNotFoundError(
        f"File INT8 tidak ditemukan: {ONNX_INT8}\n"
        f"Jalankan step2_quantize_int8.py terlebih dahulu!"
    )

# ── Verifikasi output shape ──
print("[INFO] Memuat model INT8...")
sess       = ort.InferenceSession(ONNX_INT8, providers=['CPUExecutionProvider'])
test_input = np.zeros((2, 3, 224, 224), dtype=np.float32)
out        = sess.run(None, {"eye_crops": test_input})
print(f"[INFO] Output shape: {out[0].shape}  ← harusnya (2, 512)")

# ── Warmup ──
print("[INFO] Warmup 5 runs...")
for _ in range(5):
    sess.run(None, {"eye_crops": test_input})

# ── Benchmark 30 runs ──
print("[INFO] Benchmark 30 runs di CPU...")
times = []
for _ in range(30):
    t = time.perf_counter()
    sess.run(None, {"eye_crops": test_input})
    times.append((time.perf_counter() - t) * 1000)

print(f"\n[BENCH] Rata-rata : {sum(times)/len(times):.1f} ms")
print(f"[BENCH] Minimum   : {min(times):.1f} ms")
print(f"[BENCH] Maksimum  : {max(times):.1f} ms")
print(f"[BENCH] Estimasi FPS: {1000 / (sum(times)/len(times)):.1f} fps")
print("\n[DONE] Paste hasil BENCH ini ke mentor untuk analisis selanjutnya.")

[INFO] Memuat model INT8...
[INFO] Output shape: (2, 512)  ← harusnya (2, 512)
[INFO] Warmup 5 runs...
[INFO] Benchmark 30 runs di CPU...

[BENCH] Rata-rata : 39.0 ms
[BENCH] Minimum   : 28.2 ms
[BENCH] Maksimum  : 43.4 ms
[BENCH] Estimasi FPS: 25.7 fps

[DONE] Paste hasil BENCH ini ke mentor untuk analisis selanjutnya.
